<a href="https://colab.research.google.com/github/tensorbytes0202/A/blob/main/a.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/cs-460-muffin-vs-chihuahua-classification-challenge.zip"
extract_path = "/content/dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset Extracted!")

Dataset Extracted!


In [ ]:
import os
import random
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image
from tensorflow.keras import layers,models

In [ ]:
print("TensorFlow version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.19.0
Available GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
train_dir = '/content/dataset/train'
test_dir = '/content/dataset/kaggle_test_final'

In [ ]:
muffin_count = len(os.listdir(os.path.join(train_dir, "muffin")))
chihuahua_count = len(os.listdir(os.path.join(train_dir, "chihuahua")))

total = muffin_count + chihuahua_count

print("Muffin images:", muffin_count)
print("Chihuahua images:", chihuahua_count)
print("Total images:", total)


print("\nClass distribution:")
print("Muffin:", muffin_count/total*100, "%")
print("Chihuahua:", chihuahua_count/total*100, "%")

Muffin images: 2174
Chihuahua images: 2559
Total images: 4733

Class distribution:
Muffin: 45.93281216987112 %
Chihuahua: 54.06718783012888 %


In [ ]:
train_datagen = ImageDataGenerator(
    rescale = 1./255,
    rotation_range=25,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.2,
    shear_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.8,1.2],
    validation_split=0.2
)

In [ ]:
validation_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

In [ ]:
train_generator = validation_datagen.flow_from_directory(
    train_dir,
    target_size=(256,256),
     batch_size=32,
    class_mode='binary',
    subset='validation'

)

Found 945 images belonging to 2 classes.


In [ ]:
model = models.Sequential([
    layers.Conv2D(32,(3,3),padding='same',input_shape=(256,256,3)),
    layers.Activation('relu'),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64,(3,3),padding='same'),
    layers.Activation('relu'),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128,(3,3),padding='same'),
    layers.Activation('relu'),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),

    layers.Dense(128),
    layers.Activation('relu'),
    layers.Dropout(0.5),

    layers.Dense(1,activation='sigmoid')
    ])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(
    optimizer=optimizer,
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 256, 256, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 256, 256, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 128, 128, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 128, 128, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 128, 128, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 64, 64, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 64, 64, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 131072)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    16,777,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,870,721 (64.36 MB)

 Trainable params: 16,870,721 (64.36 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
validation_generator = validation_datagen.flow_from_directory(
    train_dir,
    target_size=(256, 256),
    batch_size=32,
    class_mode='binary',
    subset='validation' # Assuming validation_datagen is meant for validation split
)

history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=10
)

Found 945 images belonging to 2 classes.
Epoch 1/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 14s 455ms/step - accuracy: 0.7481 - loss: 0.5152 - val_accuracy: 0.8063 - val_loss: 0.4153
Epoch 2/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 12s 414ms/step - accuracy: 0.8487 - loss: 0.3634 - val_accuracy: 0.8899 - val_loss: 0.2585
Epoch 3/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 11s 388ms/step - accuracy: 0.8772 - loss: 0.2952 - val_accuracy: 0.8984 - val_loss: 0.2285
Epoch 4/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 12s 389ms/step - accuracy: 0.9206 - loss: 0.2129 - val_accuracy: 0.9556 - val_loss: 0.1391
Epoch 5/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 11s 352ms/step - accuracy: 0.9302 - loss: 0.2003 - val_accuracy: 0.9757 - val_loss: 0.0999
Epoch 6/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 11s 353ms/step - accuracy: 0.9683 - loss: 0.0975 - val_accuracy: 0.9735 - val_loss: 0.0657
Epoch 7/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 11s 380ms/step - accuracy: 0.9778 - loss: 0.0732 - val_accuracy: 0.9841 - val_loss: 0.0412
Epoch 8/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 12s 391ms/step - a

In [ ]:
# Accuracy
train_acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

# Loss
train_loss = history.history['loss']
val_loss = history.history['val_loss']

epochs = range(1, len(train_acc) + 1)

plt.figure(figsize=(12,5))

# Accuracy plot
plt.subplot(1,2,1)
plt.plot(epochs, train_acc, marker='o', label='Training Accuracy')
plt.plot(epochs, val_acc, marker='o', label='Validation Accuracy')
plt.title("Training vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)

# Loss plot
plt.subplot(1,2,2)
plt.plot(epochs, train_loss, marker='o', label='Training Loss')
plt.plot(epochs, val_loss, marker='o', label='Validation Loss')
plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

plt.show()